In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
# Load train and test
train_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\train_encoded3.csv")
test_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded3.csv")

In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16016 entries, 0 to 16015
Data columns (total 26 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   profit_per_order                          16016 non-null  float64
 1   order_item_discount                       16016 non-null  float64
 2   order_item_product_price                  16016 non-null  float64
 3   order_item_profit_ratio                   16016 non-null  float64
 4   order_item_quantity                       16016 non-null  float64
 5   sales                                     16016 non-null  float64
 6   order_profit_per_order                    16016 non-null  float64
 7   shipping_mode                             16016 non-null  int64  
 8   distance_normalized                       16016 non-null  float64
 9   order_to_shipment_days                    16016 non-null  float64
 10  order_shipping_time               

In [4]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")

# 1. Prepare Data
X_train = train_df.drop('target', axis=1)
y_train = train_df['target'].astype('category')

X_test = test_df.drop('target', axis=1)
y_test = test_df['target'].astype('category')

label_mapping = {-1: 0, 0: 1, 1: 2}

y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

# 2. Models to Evaluate
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
    "LightGBM": LGBMClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM (RBF Kernel)": SVC(probability=True),
    "Naive Bayes": GaussianNB()
}

# 3. Cross Validation Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. Evaluate all models using Accuracy and F1 Score
results = []
for name, model in models.items():
    pipeline = Pipeline([("scaler", StandardScaler()), ("clf", model)])

    acc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro')  # for multiclass

    results.append({
        "Model": name,
        "CV Accuracy": acc_scores.mean(),
        "CV F1 Macro": f1_scores.mean()
    })

# 5. Show results
results_df = pd.DataFrame(results).sort_values(by='CV F1 Macro', ascending=False)
print(results_df)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001597 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4363
[LightGBM] [Info] Number of data points in the train set: 12812, number of used features: 25
[LightGBM] [Info] Start training from score -1.164088
[LightGBM] [Info] Start training from score -1.239350
[LightGBM] [Info] Start training from score -0.920750
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001187 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4366
[LightGBM] [Info] Number of data points in the train set: 12813, number of used features: 25
[LightGBM] [Info] Start training from score -1.164166
[LightGBM] [Info] Start training from score -1.239159
[LightGBM] [Info] Start training from score -0.920828
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000839 sec

Based on our CV results, Random Forest is the best single choice right now (highest CV F1 Macro = 0.746443).
It’s a robust, low-risk pick for production and a great candidate to do your focused hyperparameter optimization on first.

That said — XGBoost and LightGBM are close (XGBoost F1 = 0.729060, LightGBM F1 = 0.722090). If you can afford the extra compute/time, tune XGBoost/LightGBM too — they often overtake Random Forest after careful tuning.